In [1]:
from neo4j import GraphDatabase
from langchain_openai import OpenAIEmbeddings

In [2]:
neo4j_uri = "bolt://localhost:7687"
neo4j_username = "neo4j"
neo4j_password = "neo4jpassword"

driver = GraphDatabase.driver(
    neo4j_uri, auth=(neo4j_username, neo4j_password)
)

print("Connected to Neo4j")

Connected to Neo4j


In [3]:
embedder = OpenAIEmbeddings(model="text-embedding-3-small")

In [4]:
ontology_labels = [
    "Dataset", "DataType", "SpatialDomain", "DataAttribute",
    "VisualizationTechnique", "RenderingMethod", "FeatureExtractionMethod",
    "VisualizationTask", "UserGoal", "Feature", "Variable", "Phenomenon",
    "SoftwareSystem", "Algorithm", "Library"
]

In [5]:
def get_all_entities():
    query = """
    MATCH (n)
    WHERE n:Dataset OR n:DataType OR n:SpatialDomain OR n:DataAttribute OR
          n:VisualizationTechnique OR n:RenderingMethod OR n:FeatureExtractionMethod OR
          n:VisualizationTask OR n:UserGoal OR n:Feature OR n:Variable OR
          n:Phenomenon OR n:SoftwareSystem OR n:Algorithm OR n:Library
    RETURN n
    """

    with driver.session() as session:
        return [record["n"] for record in session.run(query)]
    

entities = get_all_entities()
print("Entities found:", len(entities))
for e in entities[:5]:
    print(e)

Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `VisualizationTechnique` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=4, column=13, offset=102>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 102, 'line': 4, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (n)\n    WHERE n:Dataset OR n:DataType OR n:SpatialDomain OR n:DataAttribute OR\n          n:VisualizationTechnique OR n:RenderingMethod OR n:FeatureExtractionMethod OR\n          n:VisualizationTask OR n:UserGoal OR n:Feature OR n:Variable OR\n          n:Phenomenon OR n:SoftwareSystem OR n:Algorithm OR n:Library\

Entities found: 2023
<Node element_id='4:109dd41d-7ea9-483d-9ea4-4027bfe2f42d:1' labels=frozenset({'Dataset'}) properties={'name': 'Volume Visualization: Principles And Advances', 'id': 'Volume Visualization: Principles And Advances'}>
<Node element_id='4:109dd41d-7ea9-483d-9ea4-4027bfe2f42d:12' labels=frozenset({'Dataset'}) properties={'name': 'Volume Data', 'id': 'Volume Data'}>
<Node element_id='4:109dd41d-7ea9-483d-9ea4-4027bfe2f42d:15' labels=frozenset({'Algorithm'}) properties={'name': 'Optimization Methods For Volume Rendering', 'id': 'Optimization Methods For Volume Rendering'}>
<Node element_id='4:109dd41d-7ea9-483d-9ea4-4027bfe2f42d:24' labels=frozenset({'Phenomenon'}) properties={'name': 'Amorphous Phenomena', 'id': 'Amorphous Phenomena'}>
<Node element_id='4:109dd41d-7ea9-483d-9ea4-4027bfe2f42d:30' labels=frozenset({'Algorithm'}) properties={'name': 'Constructive Solid Modeling', 'id': 'Constructive Solid Modeling'}>


In [6]:
def get_all_relationships():
    query = """
    MATCH (a)-[r]->(b)
    WHERE a.id IS NOT NULL AND b.id IS NOT NULL
      AND NOT a:Chunk AND NOT b:Chunk
    RETURN a, r, b
    """

    with driver.session() as session:
        return list(session.run(query))

rels = get_all_relationships()
print(f"\nTotal relationships found: {len(rels)}")
for item in rels[:5]:
    print(item["a"]["name"], "--", item["r"].type, "-->", item["b"]["name"])




Total relationships found: 7468
Volume Visualization: Principles And Advances -- CONTAINS_FEATURE --> Volume Graphics
Volume Visualization: Principles And Advances -- CONTAINS_FEATURE --> Volume Graphics
Volume Visualization: Principles And Advances -- CONTAINS_FEATURE --> Irregular Grid Rendering
Volume Visualization: Principles And Advances -- CONTAINS_FEATURE --> Irregular Grid Rendering
Volume Visualization: Principles And Advances -- CONTAINS_FEATURE --> Global Illumination Of Volumetric Data


In [7]:
def get_chunks_mentioning(name):
    query = """
    MATCH (c:Chunk)-[:MENTIONS]->(e)
    WHERE toLower(e.name) CONTAINS toLower($n)
    RETURN c.id AS chunk_id, c.text AS text, e.name AS entity
    """

    with driver.session() as session:
        return list(session.run(query, {"n": name}))

results = get_chunks_mentioning("volume rendering")
print(f"\nChunks mentioning 'volume rendering': {len(results)}")
for r in results[:3]:
    print("---")
    print(r["text"][:400], "...")


Chunks mentioning 'volume rendering': 322
---
Volume V isualization: Principles and Advances
Arie E. Kaufman
Center for Visual Computing
and Computer Science Department
State University of NewY ork at StonyB rook
StonyB rook, NY 11794-4400
ari@cs.sunysb.edu
http://www.cs.sunysb.edu/Äari
Abstract
This paper is a survey ofv olume visualization.It includes an introduction to volumetric data; surface
rendering techniques for volume data; volume r ...
---
Volume V isualization: Principles and Advances
Arie E. Kaufman
Center for Visual Computing
and Computer Science Department
State University of NewY ork at StonyB rook
StonyB rook, NY 11794-4400
ari@cs.sunysb.edu
http://www.cs.sunysb.edu/Äari
Abstract
This paper is a survey ofv olume visualization.It includes an introduction to volumetric data; surface
rendering techniques for volume data; volume r ...
---
rendering techniques for volume data; volume rendering techniques, including image-order,o bject-
order,a nd domain techniques; optimiz

In [8]:
def vector_search(query_text, top_k=5):
    embedding = embedder.embed_query(query_text)

    query = """
    CALL db.index.vector.queryNodes(
        'text_embeddings',
        $top_k,
        $embedding
    ) YIELD node, score
    RETURN node, score
    """

    with driver.session() as session:
        return list(session.run(query, {"embedding": embedding, "top_k": top_k}))

def explore_graph_entity(name):
    query = """
    MATCH (e {name: $name})-[r]-(n)
    RETURN e, r, n
    """
    with driver.session() as session:
        return list(session.run(query, {"name": name}))


print("Extraction module loaded ✔")

Extraction module loaded ✔


In [10]:
vs = vector_search("isosurface extraction vs volume rendering")

for r in vs[:3]:
    print(r["score"], r["node"]["text"][:300], "...")

0.8016963005065918 in a single 2D image.Volume rendering convey more information than surface rendering images, but at
the cost of increased algorithm complexity,and consequently increased rendering times.To improve
interactivity in volume rendering, manyo ptimization methods as well as several special-purpose volume
 ...
0.7983102798461914 approximating a surface contained within the data using geometric primitives. When volumetric data
are visualized using a surface rendering technique, a dimension of information is essentially lost.In
response to this, volume rendering techniques were developed that attempt to capture the entire 3D  ...
0.7981677055358887 2. The volumetricdataaredirectlyrenderedwithoutthein-
termediateconversionstep.This isreferredto as direct
volume rending(DVR) 20
 88
 100.
The formerassumes(i)thata setofextractableiso-surfaces
exists,and (ii)thatwiththeinﬁnitelythinsurface thepoly-
gon mesh models the trueobjectstructuresat reason ...


In [12]:
vs = vector_search("how can i code volume rendering")

for r in vs[:10]:
    print(r["score"], r["node"]["text"][:300], "...")

0.8337211608886719 Ruediger.W estermann@informatik.uni-stuttgart.de

He wlett-PackardLaboratories,PaloAlto,CA 94304-1126,USA,
e-Mail:craig_wittenbrink@hpl.hp.com
volume renderingincategories.Accelerationtechniquesto
speedup therenderingprocessinsection4.Section3 and 4
area modiﬁed versionoftutorialnotesfrom R. Y agel ...
0.8263282775878906 Volume Visualization -18- Arie Kaufman
rendering algorithm this data structure supports classi®cation and rendering a 2563 voxelv olume in
three seconds.The method extends to support mixed volumes and geometry and is parallelizable [62].
One obvious optimization for both discrete and continuous ray  ...
0.8228154182434082 in a single 2D image.Volume rendering convey more information than surface rendering images, but at
the cost of increased algorithm complexity,and consequently increased rendering times.To improve
interactivity in volume rendering, manyo ptimization methods as well as several special-purpose volume
 ...
0.818300724029541 Volume Vis

In [13]:
vs = vector_search("if i have amr data in cells, what should i use to visualize")

for r in vs[:10]:
    print(r["score"], r["node"]["text"][:300], "...")

0.7599425315856934 Volume Visualization -11- Arie Kaufman
from confocal microscope data, while Figure 2 (b) is a maximum projection of the same cell.Figure 2
wa sg enerated using the PARC algorithm, which is described in Section 5.As opposed to a surface
projection, a maximum projection is capable of revealing some in ...
0.7469878196716309 Volume Visualization -11- Arie Kaufman
from confocal microscope data, while Figure 2 (b) is a maximum projection of the same cell.Figure 2
wa sg enerated using the PARC algorithm, which is described in Section 5.As opposed to a surface
projection, a maximum projection is capable of revealing some in ...
0.7056326866149902 example isthevisiblehuman projectwhere thistechnique
hasbeen appliedtoa male and a femalecadaver.
Microscopicanalysisisyetanotherapplicationﬁeld of
volume rendering.W ithconfocalmicroscopes,itispossible
togethigh-resolutionopticalslicesofa microscopicobject
withouthavingtodisturbthespecimen.
Geoseis ...
0.7048811912536621 Volume Vi